# AIML-01 - Exploratory Data Analysis

**Education/portfolio project. Not a production fraud system.**

EDA for the Real-Time Fraud Detection Platform. Works on synthetic data out of the box;
drop the IEEE-CIS CSVs into `data/` to run it on the real dataset.


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src.common import data_loader
from src.features import feature_engineering as fe
from src.models import graph_builder
df = data_loader.load_raw()
print('source:', df.attrs.get('source'), '| shape:', df.shape)
df.head()


## 1. Class imbalance
Fraud is rare - this is why we use PR-AUC, not accuracy.


In [ ]:
rate = df['isFraud'].mean()
print(f'fraud rate: {rate:.3%}')
df['isFraud'].value_counts().plot.bar(title='class counts (0=legit, 1=fraud)'); plt.show()


## 2. Transaction amount by class


In [ ]:
import numpy as np
legit = df.loc[df.isFraud==0,'TransactionAmt']
fraud = df.loc[df.isFraud==1,'TransactionAmt']
plt.hist(np.log1p(legit), bins=50, alpha=0.5, label='legit', density=True)
plt.hist(np.log1p(fraud), bins=50, alpha=0.5, label='fraud', density=True)
plt.xlabel('log(1+amount)'); plt.legend(); plt.title('amount distribution'); plt.show()


## 3. Engineered + graph features separate the classes


In [ ]:
feats = graph_builder.attach_ring_features(fe.engineer(df))
for col in ['device_change_flag','addr_change_flag','txn_velocity_card_24h','ring_degree','ring_fraud_rate']:
    print(col, '-> mean by class:', feats.groupby('isFraud')[col].mean().round(3).to_dict())


## 4. Largest suspicious fraud rings


In [ ]:
graph_builder.ring_summary(feats, top=10)


## Takeaways
- Severe imbalance -> PR-AUC + class weighting/SMOTE.
- `device_change_flag` and `ring_degree` carry strong signal.
- Rings show up as larger connected components with elevated fraud rates.
